# Question 9 - Standard AE vs VAE Bottleneck (MNIST, 784 -> 2D)
1. **Standard AE encoder** output layer: `Dense(2)` (linear), after 784 -> 256 -> 128.
2. **VAE encoder** outputs two vectors:
   - **z_mean ($\mu$)**: mean of the latent Gaussian, shape **(2,)**
   - **z_log_var ($\log\sigma^2$)**: log-variance of the latent Gaussian, shape **(2,)**

   (With a batch: (B, 2) each.) The latent z is then sampled with $z = \mu + e^{0.5\log\sigma^2}\epsilon$.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
(x_train, _), _ = keras.datasets.mnist.load_data()
x = x_train[:16].reshape(-1, 784).astype("float32") / 255.0

In [ ]:
# 1. Standard autoencoder encoder: 784 -> 256 -> 128 -> 2
ae_encoder = keras.Sequential([
    layers.Input((784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(2)                       # output layer: 2-D latent code
])
ae_encoder.summary()
print("AE latent shape:", ae_encoder(x).shape)

In [ ]:
# 3. VAE encoder: two separate linear output layers
class VAEEncoder(keras.Model):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.h1 = layers.Dense(256, activation='relu')
        self.h2 = layers.Dense(128, activation='relu')
        self.z_mean = layers.Dense(latent_dim, name="z_mean")
        self.z_log_var = layers.Dense(latent_dim, name="z_log_var")

    def call(self, x):
        h = self.h2(self.h1(x))
        return self.z_mean(h), self.z_log_var(h)

z_mean, z_log_var = VAEEncoder()(x)
print("z_mean shape:", z_mean.shape, " z_log_var shape:", z_log_var.shape)